# Whale Migration & Sea Ice Analysis

This notebook visualizes whale migration movement and sea ice conditions in the Arctic, highlighting anomalous years where extreme sea ice conditions correlate with deviations in historical migration patterns. 

The aim is to explore relationship between whale movement patterns and sea ice concentration at coarse resolution (25km). It is a first validation of whether the type of forecasts could be relevant to anticipate whale species distribution during seasonal migrations. It also adds some of the ship tracking data, highlighting areas where ships come in to close contact with whales. 
Later the exact user needs and requirements should be further validated with a domain expert.

We will plot OSISAF sea ice concentration data and overlay open source whale telemetry data for three years, including one year of observed anomalous whale movement (to get an indication of anomalies). For the most recent year, ship tracking data for a selection of ships is also added

The notebook covers
- Loading and inspecting data
    - **Whale telemetry data**: Bowhead whale locations from West Greenland
    - **Sea ice data**: OSISAF sea ice concentration data for the Arctic
    - **Ship track data**: AIS records for selected ships
    - **Temporal focus**: Summer months (July-September) across multiple years
- Generating animated visualisations

## 1. Getting the data 
This notebook will load three different datasets:
- Whale telemetry data $^1$
- Ship tracking data $^2$
- OSISAF sea ice concentration data

$^1$ Chambault, Philippine; Kovacs, Kit M; Lydersen, Christian et al. (2022). "Future seasonal changes in habitat for Arctic whales during predicted ocean warming", https://doi.org/10.1126/sciadv.abn2422.

$^2$ Data from Global Fishing Watch: https://globalfishingwatch.org/map/vessel-search

### Create a config file

There are three accompanying config files for this notebook, all in the `cryocast/config/` folder

- `demo_whale_corridors.yaml` - specifies dataset configuration and links to other relevant config files
- `notebooks/whale_telemetry_csv` - specifies parameters for filtering whale data
- `notebooks/whale_visualization` - specifies parameters for visualizing data

You should start by creating a config file called `cryocast/config/whale_corridors.local.yaml` containing the following data:

```
defaults:
  - demo_whale_corridors
  - _self_

base_path: /LOCAL/PATH/WHERE/YOU/WANT/TO/STORE/DATA
```

### Download whale telemetry data

You need to download the whale telemetry data from Dryad [https://doi.org/10.5061/dryad.tqjq2bw2c](https://doi.org/10.5061/dryad.tqjq2bw2c).

This **must** be stored at `${base_path}/data/notebooks/whale_corridors/`.

### Download sea ice concentration data

We use the `cryocast datasets create` command to download OSISAF sea ice concentration data.

In [ ]:
!cryocast datasets create --config-name whale_corridors.local

## 2. Setup and Imports

Import all necessary libraries for data processing, visualization, and analysis. This includes scientific computing libraries (numpy, pandas, xarray), visualization tools (matplotlib, cartopy), and the CryoCast framework components.

In [ ]:
from pathlib import Path

# Third-party libraries
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import hydra
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from s2sphere import CellId

# Jupyter/interactive videos
from io import BytesIO
from urllib.request import urlopen
from cartopy.io.img_tiles import GoogleWTS
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.lines import Line2D
from matplotlib.collections import LineCollection
from sklearn.neighbors import BallTree
from matplotlib.colors import ListedColormap
from IPython.display import Video



In [ ]:
with hydra.initialize(config_path="../cryocast/config", version_base=None):
    config = hydra.compose(config_name="whale_corridors.local")

## 3. Inspect & Filter the Whale Data

Ensure you have downloaded the whale telemetry CSV dataset manually and saved it to the path specified in whale_telemetry config file.

In [ ]:
# Load the CSV file
csv_path = Path(config.notebooks.whale_telemetry.input.csv.path)
print(f"Loading telemetry data from {csv_path.resolve()}")

# Load CSV with specified separator
separator = config.notebooks.whale_telemetry.input.csv.separator
whale_data = pd.read_csv(csv_path, sep=separator)

# Apply column mapping from config
column_mapping = config.notebooks.whale_telemetry.columns
whale_data = whale_data.rename(columns=column_mapping)

# Convert dateTime to datetime
whale_data['dateTime'] = pd.to_datetime(whale_data['dateTime'], format='%d/%m/%Y %H:%M')

# Extract month and year from dateTime
whale_data['month'] = whale_data['dateTime'].dt.month
whale_data['year'] = whale_data['dateTime'].dt.year

# Basic inspection of whale_data
print("Columns:", list(whale_data.columns))
print("Shape:", whale_data.shape)
print("\nRows per species:")
print(whale_data['species'].value_counts())

We can see that we have the most data for the Narwhal species.

Narwhals can survive in mean SIC conditions up to 98% (but rely on cracks and leads to breath)

Bowhead avoid SIC above 65%, prefer medium and first-year ice, prefer small floe size to avoid entrapment in winter but seek SIC above 65% during summer to avoid predators; they can break through ice fore breathing.

Because at the current resolution of CryoCast forecasts we are not able to predict the location of cracks and leads, we will focus on Bowhead whales, where we are more likely to identify a SIC threshold to predict whale location.

In [ ]:
bowhead_data = whale_data[whale_data['species'] == 'Bw']

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    
# Year distribution
year_counts = bowhead_data.groupby('year').size()
ax1.bar(year_counts.index, year_counts.values, color='skyblue', alpha=0.7, edgecolor='navy')
ax1.set_title(f'Bowhead - Data Points per Year')
ax1.set_xlabel('Year')
ax1.set_ylabel('Number of Data Points')
ax1.grid(True, alpha=0.3)

# Month distribution
month_counts = bowhead_data.groupby('month').size()
month_names = [pd.Timestamp(2020, month, 1).strftime('%B') for month in month_counts.index]
ax2.bar(range(len(month_counts)), month_counts.values, color='lightcoral', alpha=0.7, edgecolor='darkred')
ax2.set_title(f'Bowhead - Data Points per Month')
ax2.set_xlabel('Month')
ax2.set_ylabel('Number of Data Points')
ax2.set_xticks(range(len(month_counts)))
ax2.set_xticklabels(month_names, rotation=45)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

We can see that we do not have data points for the main migration periods which are April to June. However, we might be able to see some movement still during early spring months August. We also note that we have a decent amount of data for Bowhead whales for the year of 2010.

2010 is a special year as that year the Northwest Passage was largely free of sea ice by August 10th, providing unusual access. It has been reported that Two adult male bowhead whales, one from West Greenland and one from Alaska (different stocks), entered the Northwest Passage from opposite directions. They spent over two weeks in Viscount Melville Sound in September, coming within 130 km of each other, before returning to their normal seasonal ranges. (Heide-Jørgensen, Mads Peter, et al. "The Northwest Passage opens for bowhead whales." Biology letters 8.2 (2012): 270-273.)

For comparison, we select the years 2003 and 2018 as alternative years with decent amount of data points and both more and less ice coverage respectively (relative to 2010 based on reported September minimum ice extent https://climate.copernicus.eu/sea-ice-cover-september-2024)

We will also filter telemetry data for the West side of Greenland, which is where the main migration movement is expected to happen

In [ ]:
print("Whale telemetry config:")
print(f"Species filter: {config.notebooks.whale_telemetry.filters.species}")
print(f"Side filter: {config.notebooks.whale_telemetry.filters.side}")
print(f"Start month: {config.notebooks.whale_telemetry.date_filter.start_month}")
print(f"End month: {config.notebooks.whale_telemetry.date_filter.end_month}")
print(f"Target years: {config.notebooks.whale_telemetry.date_filter.target_years}")

# Use config to filter the data
filtered_whale_data = whale_data[
    (whale_data['species'] == config.notebooks.whale_telemetry.filters.species) &
    (whale_data['side'] == config.notebooks.whale_telemetry.filters.side) &
    (whale_data['month'] >= config.notebooks.whale_telemetry.date_filter.start_month) &
    (whale_data['month'] <= config.notebooks.whale_telemetry.date_filter.end_month) &
    (whale_data['year'].isin(config.notebooks.whale_telemetry.date_filter.target_years))
]

print(f"\nFiltered records: {len(filtered_whale_data):,}")

## 4. Whale Data Visualization

Let's visualize one day of the whale dataset to get a better idea

In [ ]:
def create_separate_data_plots(whale_data_filtered: pd.DataFrame, config) -> None:
    """
    Create whale data visualization for one day from a randomly selected year.
    
    Args:
        whale_data_filtered: Filtered whale telemetry data
        config: Configuration object
    """
    # Get the first day's data for this year
    first_date = whale_data_filtered['dateTime'].min().date()
    whale_data_date = whale_data_filtered[whale_data_filtered['dateTime'].dt.date == first_date]
    
    # Create whale locations plot
    fig, ax = plt.subplots(1, 1, figsize=(12, 10), 
                          subplot_kw={'projection': ccrs.LambertAzimuthalEqualArea(
                              central_longitude=0, central_latitude=90)})
    
    # Set Arctic extent (West Greenland / Canadian Arctic)
    plot_extent = [-110, -55, 60, 85]
    ax.set_extent(plot_extent, crs=ccrs.PlateCarree())
    
    # Add coastlines and land features
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, color='black')
    ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.7)
    
    # Plot whale locations if data exists
    if not whale_data_date.empty:
        # Plot whale locations with different colors per whale ID
        unique_whales = whale_data_date['id'].unique()
        colors = plt.cm.tab10(np.linspace(0, 1, len(unique_whales)))
        
        for i, whale_id in enumerate(unique_whales):
            whale_points = whale_data_date[whale_data_date['id'] == whale_id]
            ax.scatter(whale_points['lon'], whale_points['lat'], 
                       c=[colors[i]], label=f'Whale {whale_id}', 
                       s=50, alpha=0.7, edgecolors='black', linewidth=0.5,
                       transform=ccrs.PlateCarree())
        
        # Add legend
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10,
                  title='Whale IDs', title_fontsize=11, framealpha=0.9)
        
        # Add data summary info
        ax.text(0.02, 0.98, f'Total Whales: {len(unique_whales)}\nTotal Points: {len(whale_data_date)}', 
                transform=ax.transAxes, fontsize=9, style='italic',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8),
                verticalalignment='top')
    else:
        ax.text(0.5, 0.5, 'No whale data for this year', 
                ha='center', va='center', transform=ax.transAxes, fontsize=14)
    
    # Set title
    ax.set_title(f'Whale Telemetry Locations - {first_date}', 
                 fontsize=16, fontweight='bold', pad=20)
    
    # Add informative subtitle
    ax.text(0.5, 0.95, 'Bowhead Whale (Bw) | West Greenland Side', 
            transform=ax.transAxes, ha='center', va='top', 
            fontsize=10, style='italic', bbox=dict(boxstyle='round,pad=0.3', 
            facecolor='white', alpha=0.8))
    
    # Add gridlines
    ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                linewidth=0.7, alpha=0.8, color='gray')
    
    # Add coordinate system info
    ax.text(0.02, 0.02, 'Projection: Lambert Azimuthal Equal Area\nCRS: WGS84 (EPSG:4326)', 
            transform=ax.transAxes, fontsize=8, style='italic',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
    
    plt.show()

# Filter whale data for year
whale_data_year = filtered_whale_data[filtered_whale_data['year'] == 2018]
    
create_separate_data_plots(whale_data_filtered=whale_data_year, config=config)

We can see that on this random day we have data for 5 unique whales and for some of these whales we have several data points indicating their travel trajectory.

## 5. Sea Ice Data

We will now add sea ice to the mix!

Let's load the prepared Zarr sea ice datasets using the lightning SingleDataset format. This loads OSISAF sea ice concentration data for each target year and applies sanity checks to verify data integrity.

In [ ]:
# Load sea ice datasets using SingleDataset
from cryocast.data import SingleDataset

# Get the base path from config
base_path = Path(config.base_path)

# Dictionary to store datasets for each year
sea_ice_datasets = {}

# Get target years from whale telemetry config
target_years = config.notebooks.whale_telemetry.date_filter.target_years

for year in target_years:
    dataset_name = f"nbsamp-sicnorth-osisaf-25p0km-{year}-{year}-24h-v1"
    dataset_path = base_path / "data" / "anemoi" / f"{dataset_name}.zarr"
    try:
        # Create SingleDataset using the lightning format
        # This will load the data using anemoi.datasets.data.open_dataset internally
        sea_ice_datasets[str(year)] = SingleDataset(
            name=dataset_name, input_files=[dataset_path], variables=["ice_conc"]
        )
    except Exception as e:
        print(f"Failed to load sea ice dataset for {year}: {e}")
print(sea_ice_datasets)

In [ ]:
# Get the first dataset (any year)
year, dataset = next(iter(sea_ice_datasets.items()))

# Get first day's data
first_day_data = dataset[0]  # Shape: (C, H, W)
first_day_data = first_day_data.squeeze(0)  # Remove channel dimension -> (H, W)

# Create plot
fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={'projection': ccrs.NorthPolarStereo()})

# Data’s CRS (from the manual): LAEA, pole-centered, WGS84, lon0=0
src_crs = ccrs.LambertAzimuthalEqualArea(
    central_longitude=0, central_latitude=90,
    globe=ccrs.Globe(ellipse='WGS84')
)

nrows, ncols = 432, 432
px = 25_000.0  # meters per pixel

half_w = (ncols * px) / 2.0
half_h = (nrows * px) / 2.0
extent = [-half_w, half_w, -half_h, half_h]   # [xmin, xmax, ymin, ymax] in meters
print(extent) 
ax = plt.axes(projection=src_crs)

# Plot sea ice data using imshow
im1 = ax.imshow(
    first_day_data,
    vmin=0, vmax=1,
    transform=src_crs,         
    extent=extent,             # Use the EASE2 grid extent
    origin='upper',
    cmap='Blues_r'            # Blue to white colormap (white = high concentration)
)

# Add map features
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, color='black')
ax.add_feature(cfeature.LAND, alpha=0.7, zorder=0)

plt.show()

## 6. Generate animation

We now overlay whale telemetry data on top of SIC to generate an animation.

First we set up some helper functions

In [ ]:
class ArcGISWorldImagery(GoogleWTS):
    """High-resolution satellite imagery from ArcGIS (no API key required)."""
    def _image_url(self, tile):
        x, y, z = tile
        return f'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}'


_github_logo_cache = None

def get_github_logo():
    global _github_logo_cache
    if _github_logo_cache is None:
        url = "https://github.githubassets.com/images/modules/logos_page/GitHub-Mark.png"
        with urlopen(url) as response:
            _github_logo_cache = plt.imread(BytesIO(response.read()))
    return _github_logo_cache


def track_to_colored_collection(track, max_gap, color_fn=None, zorder=19):
    """Build a LineCollection for a track, fading transparent→opaque.

    color_fn: callable(t) -> (r, g, b, a) where t∈[0,1] (0=oldest, 1=newest).
              Defaults to whale colors: cyan (oldest, transparent) → blue (newest, opaque).
    """
    if color_fn is None:
        color_fn = lambda t: (0.0, 1.0 - t, 1.0, 0.1 + 0.9 * t)
    lons = track['lon'].values
    lats = track['lat'].values
    times = track['dateTime'].values
    try:
        duration = float((times[-1] - times[0]) / np.timedelta64(1, 'ns'))
    except Exception:
        duration = 1.0
    segments, colors = [], []
    for i in range(len(lons) - 1):
        if pd.Timestamp(times[i + 1]) - pd.Timestamp(times[i]) > max_gap:
            continue
        segments.append([[lons[i], lats[i]], [lons[i + 1], lats[i + 1]]])
        t = float((times[i] - times[0]) / np.timedelta64(1, 'ns')) / duration if duration > 0 else 1.0
        colors.append(color_fn(t))
    return LineCollection(segments, colors=colors, linewidth=2,
                          transform=ccrs.PlateCarree(), zorder=zorder)


def ship_tracks_for_date(frame_date, ship_df, ship_window, max_gap):
    """Return a list of LineCollections and current-position lists for the ship window."""
    ship_color_fn = lambda t: (1.0, 0.65 * (1.0 - t), 0.0, 0.1 + 0.9 * t)  # orange→red
    frame_date_utc = frame_date.tz_localize('UTC') if frame_date.tzinfo is None else frame_date
    window_start = frame_date_utc - ship_window
    visible = ship_df[
        (ship_df['dateTime'] > window_start) &
        (ship_df['dateTime'].dt.date <= frame_date.date())
    ]
    if visible.empty:
        return [], [], []
    lcs = []
    current_lons, current_lats = [], []
    for mmsi, track in visible.groupby('mmsi'):
        track = track.sort_values('dateTime')
        lc = track_to_colored_collection(track, max_gap, color_fn=ship_color_fn, zorder=10)
        lcs.append(lc)
        lons = track['lon'].values
        lats = track['lat'].values
        current_lons.append(float(lons[-1]))
        current_lats.append(float(lats[-1]))
    return lcs, current_lons, current_lats


def find_proximity_events(whale_df, ship_df, threshold_km=50):
    """Return a DataFrame of dates when a whale and ship were within threshold_km."""
    events = []
    ship_df = ship_df.copy()
    ship_df['_date'] = ship_df['dateTime'].dt.date
    for date, whales in whale_df.groupby(whale_df['dateTime'].dt.date):
        ships_day = ship_df[ship_df['_date'] == date]
        if ships_day.empty or whales.empty:
            continue
        ship_coords = np.radians(ships_day[['lat', 'lon']].values)
        whale_coords = np.radians(whales[['lat', 'lon']].values)
        tree = BallTree(ship_coords, metric='haversine')
        threshold_rad = threshold_km / 6371.0
        indices, dists = tree.query_radius(whale_coords, r=threshold_rad, return_distance=True)
        for w_idx, (idxs, ds) in enumerate(zip(indices, dists)):
            for s_idx, dist in zip(idxs, ds):
                events.append({
                    'date': date,
                    'whale_id': whales.iloc[w_idx]['id'],
                    'whale_lon': float(whales.iloc[w_idx]['lon']),
                    'whale_lat': float(whales.iloc[w_idx]['lat']),
                    'ship_mmsi': str(ships_day.iloc[int(s_idx)]['mmsi']),
                    'ship_lon': float(ships_day.iloc[int(s_idx)]['lon']),
                    'ship_lat': float(ships_day.iloc[int(s_idx)]['lat']),
                    'distance_km': float(dist * 6371.0),
                })
    if not events:
        return pd.DataFrame(columns=['date', 'whale_id', 'whale_lon', 'whale_lat',
                                     'ship_mmsi', 'ship_lon', 'ship_lat', 'distance_km'])
    df = pd.DataFrame(events)
    df = df.sort_values('distance_km').drop_duplicates(subset=['date', 'whale_id', 'ship_mmsi'])
    return df.reset_index(drop=True)


def create_animation(sea_ice_dataset: 'SingleDataset', whale_data_year: pd.DataFrame, year: str,
                     mode: str = 'lines',
                     ship_df: pd.DataFrame = None, ship_window: int = 7, whale_window: int = 7,
                     proximity_threshold_km: float = 50, show_whales: bool = True) -> 'matplotlib.animation.FuncAnimation':
    """
    Create animation for a single year's dataset.

    Args:
        sea_ice_dataset: Single year's sea ice dataset
        whale_data_year: Filtered whale telemetry data for this year
        year: Year as string for title
        mode: 'points' — dots with a 5-day colour trail (red=current → yellow=oldest);
              'lines'  — fading LineCollections with optional ship tracks and proximity highlighting
        ship_df: Ship AIS DataFrame (lines mode only)
        ship_window: Days of ship history to show (lines mode only)
        whale_window: Days of whale history to show (lines mode only)
        proximity_threshold_km: Ring-alert distance in km (lines mode only)
        show_whales: Whether to draw whale tracks/markers (lines mode only)

    Returns:
        FuncAnimation object
    """
    sea_ice_dataset = sea_ice_dataset[year]
    whale_data_year = whale_data_year[whale_data_year['year'] == int(year)]
    
    total_timesteps = len(sea_ice_dataset)
    if total_timesteps == 0:
        return None

    n = 256
    ice_rgba = np.ones((n, 4))
    ice_rgba[:, 3] = np.power(np.linspace(0, 1, n), 0.5) * 0.95
    ice_cmap = ListedColormap(ice_rgba)

    max_gap = pd.Timedelta(days=2)

    # Cache date column once — used for per-frame filtering in both modes
    _whale_dates = whale_data_year['dateTime'].dt.date

    # --- points-mode helpers ---
    if mode == 'points':
        def dot_colour(days_back):
            t = days_back / whale_window
            return (1.0, t, 0.0)

        def dot_size(days_back):
            return max(20, 120 - days_back * 18)

        def get_daily_whale_positions(df, target_date, days_back=0):
            if days_back > 0:
                target_date = target_date - pd.Timedelta(days=days_back)
            day_data = df[_whale_dates == target_date.date()]
            if day_data.empty:
                return pd.DataFrame()
            return day_data.groupby('id').tail(1)

    # --- lines-mode setup ---
    if mode == 'lines':
        ship_window = pd.Timedelta(days=ship_window)
        whale_window = pd.Timedelta(days=whale_window)

        proximity_by_date = {}
        if show_whales and ship_df is not None and not ship_df.empty:
            prox_df = find_proximity_events(whale_data_year, ship_df, threshold_km=proximity_threshold_km)
            if not prox_df.empty:
                print(f"Found {len(prox_df)} proximity event(s) on {prox_df['date'].nunique()} day(s)")
                for date, grp in prox_df.groupby('date'):
                    proximity_by_date[date] = grp

    # --- figure setup ---
    fig, ax = plt.subplots(1, 1, figsize=(12, 10),
                           subplot_kw={'projection': ccrs.LambertAzimuthalEqualArea(
                               central_latitude=90, central_longitude=0)})

    plot_extent = [-110, -55, 60, 85]
    ax.set_extent(plot_extent, crs=ccrs.PlateCarree())
    ax.add_image(ArcGISWorldImagery(cache=True), 4)
    ax.gridlines(draw_labels=(mode == 'points'), dms=True, x_inline=False, y_inline=False,
                 linewidth=0.5, alpha=0.6, color='white')

    # --- legend ---
    if mode == 'points':
        legend_elements = [
            Line2D([0], [0], marker='o', color='w', markerfacecolor=(1.0, 0.0, 0.0),
                   markeredgecolor='black', markersize=11, label='Current position'),
            Line2D([0], [0], marker='o', color='w', markerfacecolor=(1.0, 0.6, 0.0),
                   markeredgecolor='black', markersize=9, label='3 days ago'),
            Line2D([0], [0], marker='o', color='w', markerfacecolor=(1.0, 1.0, 0.0),
                   markeredgecolor='black', markersize=7, label='5 days ago'),
        ]
        legend_title = 'Bowhead Whale\ntelemetry'
    else:
        legend_elements = []
        if show_whales:
            legend_elements += [
                Line2D([0], [0], color='blue', linewidth=2, label='Recent whale track'),
                Line2D([0], [0], marker='o', linestyle='', markerfacecolor='blue',
                       markeredgecolor='black', markersize=9, label='Whale current position'),
            ]
        if ship_df is not None and not ship_df.empty:
            legend_elements += [
                Line2D([0], [0], color='red', linewidth=2, label='Recent ship track'),
                Line2D([0], [0], marker='o', linestyle='', markerfacecolor='red',
                       markeredgecolor='black', markersize=9, label='Ship current position'),
            ]
        if proximity_by_date:
            legend_elements.append(
                Line2D([0], [0], marker='o', linestyle='', markerfacecolor='none',
                       markeredgecolor='yellow', markeredgewidth=2.5, markersize=14,
                       label=f'Whale/ship proximity\n(<{proximity_threshold_km} km)'),
            )
        legend_title = 'Track data'

    legend = ax.legend(handles=legend_elements, title=legend_title,
                       title_fontsize=9, loc='upper left', fontsize=8,
                       framealpha=0.9, facecolor='white', edgecolor='gray')
    legend.set_zorder(25)

    # --- GitHub logo ---
    try:
        github_logo = get_github_logo()
        imagebox = OffsetImage(github_logo, zoom=0.07)
        ab = AnnotationBbox(imagebox, (0.02, 0.03), xycoords='axes fraction',
                            frameon=False, box_alignment=(0, 0.5), zorder=25)
        ax.add_artist(ab)
        ax.text(0.12, 0.03, 'github.com/alan-turing-institute/cryocast',
                transform=ax.transAxes, ha='left', va='center',
                fontsize=12, color='white', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.6), zorder=25)
    except Exception as e:
        print(f"Warning: Could not add GitHub logo: {e}")

    # --- lines-mode: proximity ring radius and frame expansion ---
    if mode == 'lines':
        ring_radius_px = np.sqrt(800 / np.pi) * fig.dpi / 72
        src_crs = ccrs.PlateCarree()

        def _ring_will_show(date):
            if date not in proximity_by_date or ship_df is None:
                return False
            date_utc = pd.Timestamp(date).tz_localize('UTC')
            ships_visible = ship_df[
                (ship_df['dateTime'] > date_utc - ship_window) &
                (ship_df['dateTime'].dt.date <= date)
            ]
            if ships_visible.empty:
                return False
            current_ship_pos = (
                ships_visible.sort_values('dateTime')
                .groupby('mmsi')[['lon', 'lat']].last()
            )
            for _, evt in proximity_by_date[date].iterrows():
                mmsi = evt['ship_mmsi']
                if mmsi not in current_ship_pos.index:
                    continue
                try:
                    whale_px = ax.transData.transform(
                        ax.projection.transform_point(evt['whale_lon'], evt['whale_lat'], src_crs))
                    ship_px = ax.transData.transform(
                        ax.projection.transform_point(
                            current_ship_pos.loc[mmsi, 'lon'],
                            current_ship_pos.loc[mmsi, 'lat'],
                            src_crs))
                    if np.linalg.norm(np.array(whale_px) - np.array(ship_px)) <= ring_radius_px:
                        return True
                except Exception:
                    continue
            return False

        expanded_frames = []
        for idx in range(total_timesteps):
            expanded_frames.append(idx)
            frame_date = (sea_ice_dataset.start_date + pd.Timedelta(days=idx)).date()
            if _ring_will_show(frame_date):
                expanded_frames.extend([idx] * 3)
    else:
        expanded_frames = range(total_timesteps)

    # --- animation state ---
    sea_ice_plot = None
    whale_artists = []
    date_text = None

    half = 432 * 25_000.0 / 2
    ice_extent = [-half, half, -half, half]

    def animate(frame_idx):
        nonlocal sea_ice_plot, whale_artists, date_text

        sea_ice_data = sea_ice_dataset[frame_idx].squeeze(0)
        frame_date_pd = sea_ice_dataset.start_date + pd.Timedelta(days=frame_idx)
        frame_date_str = frame_date_pd.strftime('%Y-%m-%d')

        if sea_ice_plot is not None:
            sea_ice_plot.remove()
        for artist in whale_artists:
            artist.remove()
        whale_artists.clear()
        if date_text is not None:
            date_text.remove()

        sea_ice_plot = ax.imshow(
            sea_ice_data, vmin=0, vmax=1,
            transform=ccrs.LambertAzimuthalEqualArea(central_latitude=90, central_longitude=0),
            extent=ice_extent, origin='upper', cmap=ice_cmap, zorder=2)

        if mode == 'points':
            current_whale_data = get_daily_whale_positions(whale_data_year, frame_date_pd)
            if not current_whale_data.empty:
                all_lons, all_lats, all_colors, all_sizes = [], [], [], []
                for whale_id in current_whale_data['id'].unique():
                    for days_back in range(whale_window, 0, -1):
                        trail_data = get_daily_whale_positions(whale_data_year, frame_date_pd, days_back)
                        if not trail_data.empty:
                            pt = trail_data[trail_data['id'] == whale_id]
                            if not pt.empty:
                                all_lons.append(pt['lon'].iloc[0])
                                all_lats.append(pt['lat'].iloc[0])
                                all_colors.append(dot_colour(days_back))
                                all_sizes.append(dot_size(days_back))
                    pt = current_whale_data[current_whale_data['id'] == whale_id]
                    if not pt.empty:
                        all_lons.append(pt['lon'].iloc[0])
                        all_lats.append(pt['lat'].iloc[0])
                        all_colors.append(dot_colour(0))
                        all_sizes.append(dot_size(0))
                if all_lons:
                    sc = ax.scatter(all_lons, all_lats, c=all_colors, s=all_sizes,
                                    edgecolors='black', linewidth=0.6,
                                    transform=ccrs.PlateCarree(), zorder=20)
                    whale_artists.append(sc)

        else:  # lines mode
            if ship_df is not None and not ship_df.empty:
                ship_lcs, ship_lons, ship_lats = ship_tracks_for_date(
                    frame_date_pd, ship_df, ship_window, max_gap)
                for ship_lc in ship_lcs:
                    ax.add_collection(ship_lc)
                    whale_artists.append(ship_lc)
                if ship_lons:
                    ship_sc = ax.scatter(ship_lons, ship_lats, c=['red'], s=80,
                                         edgecolors='black', linewidth=0.6,
                                         transform=ccrs.PlateCarree(), zorder=20)
                    whale_artists.append(ship_sc)

            if show_whales:
                historical_data = whale_data_year[_whale_dates <= frame_date_pd.date()]
                for whale_id in historical_data['id'].unique():
                    whale_track_full = historical_data[
                        historical_data['id'] == whale_id
                    ].sort_values('dateTime')
                    whale_track_window = whale_track_full[
                        whale_track_full['dateTime'] > frame_date_pd - whale_window
                    ]
                    if len(whale_track_window) >= 2:
                        lc = track_to_colored_collection(whale_track_window, max_gap)
                        ax.add_collection(lc)
                        whale_artists.append(lc)
                    if not whale_track_full.empty:
                        sc = ax.scatter(
                            [whale_track_full['lon'].iloc[-1]], [whale_track_full['lat'].iloc[-1]],
                            c=['blue'], s=80, edgecolors='cyan', linewidth=0.6,
                            transform=ccrs.PlateCarree(), zorder=21)
                        whale_artists.append(sc)

            if frame_date_pd.date() in proximity_by_date:
                frame_dt_utc = (frame_date_pd.tz_localize('UTC')
                                if frame_date_pd.tzinfo is None else frame_date_pd)
                ships_visible = ship_df[
                    (ship_df['dateTime'] > frame_dt_utc - ship_window) &
                    (ship_df['dateTime'].dt.date <= frame_date_pd.date())
                ]
                current_ship_pos = (
                    ships_visible.sort_values('dateTime')
                    .groupby('mmsi')[['lon', 'lat']].last()
                )
                for _, evt in proximity_by_date[frame_date_pd.date()].iterrows():
                    mmsi = evt['ship_mmsi']
                    if mmsi not in current_ship_pos.index:
                        continue
                    try:
                        whale_px = ax.transData.transform(
                            ax.projection.transform_point(evt['whale_lon'], evt['whale_lat'], src_crs))
                        ship_px = ax.transData.transform(
                            ax.projection.transform_point(
                                current_ship_pos.loc[mmsi, 'lon'],
                                current_ship_pos.loc[mmsi, 'lat'],
                                src_crs))
                        if np.linalg.norm(np.array(whale_px) - np.array(ship_px)) > ring_radius_px:
                            continue
                    except Exception:
                        continue
                    circ = ax.scatter(
                        [evt['whale_lon']], [evt['whale_lat']],
                        s=800, facecolors='none', edgecolors='yellow',
                        linewidths=3, transform=ccrs.PlateCarree(), zorder=23)
                    whale_artists.append(circ)

        date_text = ax.text(0.5, 0.97, frame_date_str, transform=ax.transAxes,
                            ha='center', va='top', fontsize=14, fontweight='bold',
                            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9), zorder=21)
        return [sea_ice_plot] + whale_artists + [date_text]

    anim = animation.FuncAnimation(fig, animate, frames=expanded_frames,
                                   interval=1000, blit=False, repeat=True)
    plt.close(fig)
    return anim


In [ ]:
sel_year = '2010'
anim = create_animation(sea_ice_datasets, filtered_whale_data, sel_year, mode='points', whale_window=5)

output_path = 'whale_corridors_2010.mp4'
writer = animation.FFMpegWriter(fps=1, bitrate=2000)
anim.save(output_path, writer=writer, savefig_kwargs={'bbox_inches': 'tight', 'pad_inches': 0.05})
plt.close('all')

Video(output_path, embed=True)

### Animation of whales / ships 

We will extend the animation to also include ship track data

The voyages data is downloaded from GFW, for August / September 2018: https://globalfishingwatch.org/data-download/datasets/public-voyages-confidence-4%3Av20220922 

In [ ]:
# Re-run S2 filtering without any API calls to recover the vessel ID list
_voyages_dir = Path(config.notebooks.whale_telemetry.input.csv.path).parent
_year = 2018
_xmin, _xmax, _ymin, _ymax = -110, -55, 60, 85

dfs = []
for month in [8, 9]:
    fname = _voyages_dir / f"voyages_{_year}{month:02d}.csv"
    if fname.exists():
        dfs.append(pd.read_csv(fname))

df_v = pd.concat(dfs, ignore_index=True)

all_tokens = pd.unique(df_v[['trip_start_anchorage_s2id', 'trip_end_anchorage_s2id']].values.ravel('K'))
arctic_s2ids = set()
for token in all_tokens:
    if not isinstance(token, str):
        continue
    try:
        ll = CellId.from_token(token).to_lat_lng()
        if _xmin <= ll.lng().degrees <= _xmax and _ymin <= ll.lat().degrees <= _ymax:
            arctic_s2ids.add(token)
    except Exception:
        pass

arctic_mask = (
    df_v['trip_start_anchorage_s2id'].isin(arctic_s2ids) |
    df_v['trip_end_anchorage_s2id'].isin(arctic_s2ids)
)
arctic_vessel_ids = df_v.loc[arctic_mask, 'vessel_id'].dropna().astype(str).unique().tolist()
print(f"Found {len(arctic_vessel_ids)} vessel IDs with Arctic voyages:")
for vid in sorted(arctic_vessel_ids):
    print(vid)


Ships are downloaded from GFW: https://globalfishingwatch.org/map/vessel-search

I selected a set of ships from the list provided by the above cell. I selected the following vessels:
    - ZELADA DESGAGNES
    - SAPUTI
    - ROSAIREADESGAGNES
    - AKADEMIK SERGEY VAVILOV
    - KIVIUQ II 
    - ARCTIC
    - CLAUDEADESGAGNES
    - SARAH DESGAGNES
    - LEEWAYODYSSEY
    - MIENA DESGAGNES


In [ ]:
data_dir = Path(config.notebooks.whale_telemetry.input.csv.path).parent

track_files = sorted(data_dir.glob("track-data*.csv"))
print("Found track files:", track_files)

if track_files:
    ship_track_csv = pd.concat(
        [pd.read_csv(f) for f in track_files], ignore_index=True
    )
    print(ship_track_csv.shape)
    print(ship_track_csv.dtypes)
    display(ship_track_csv.head())


In [ ]:
ship_track_csv['timestamp'] = pd.to_datetime(ship_track_csv['timestamp'])
ship_track_csv = ship_track_csv.sort_values(['seg_id', 'timestamp']).reset_index(drop=True)

print(f"Date range: {ship_track_csv['timestamp'].min()} → {ship_track_csv['timestamp'].max()}")
print(f"Segments:   {ship_track_csv['seg_id'].nunique()}")
print(f"Lon range:  {ship_track_csv['lon'].min():.2f} → {ship_track_csv['lon'].max():.2f}")
print(f"Lat range:  {ship_track_csv['lat'].min():.2f} → {ship_track_csv['lat'].max():.2f}")

fig, ax = plt.subplots(1, 1, figsize=(12, 10),
    subplot_kw={'projection': ccrs.LambertAzimuthalEqualArea(central_latitude=90, central_longitude=0)})
ax.set_extent([-110, -55, 60, 85], crs=ccrs.PlateCarree())
ax.add_image(ArcGISWorldImagery(cache=True), 4)
ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
             linewidth=0.5, alpha=0.6, color='white')

segments = ship_track_csv['seg_id'].unique()
cmap = plt.cm.tab20 if len(segments) <= 20 else plt.cm.hsv
colors = cmap(np.linspace(0, 1, len(segments)))

for seg, color in zip(segments, colors):
    seg_data = ship_track_csv[ship_track_csv['seg_id'] == seg]
    ax.plot(seg_data['lon'].values, seg_data['lat'].values,
            color=color, linewidth=1.2, alpha=0.85,
            transform=ccrs.PlateCarree())
    ax.scatter([seg_data['lon'].iloc[0]], [seg_data['lat'].iloc[0]],
               color=color, s=30, marker='o', transform=ccrs.PlateCarree(), zorder=5)
    ax.scatter([seg_data['lon'].iloc[-1]], [seg_data['lat'].iloc[-1]],
               color=color, s=60, marker='*', transform=ccrs.PlateCarree(), zorder=5)

ax.set_title('Ship tracks — 2018-08-16 to 2018-09-15\n(○ start, ★ end per segment)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
sel_year = '2018'


anim = create_animation(
    sea_ice_datasets,
    filtered_whale_data,
    sel_year,
    whale_window=7,
)

output_path = 'whale_corridors_2018.mp4'
writer = animation.FFMpegWriter(fps=1.5, bitrate=2000)
anim.save(output_path, writer=writer, savefig_kwargs={'pad_inches': 0.05})
plt.close('all')

Video(output_path, embed=True)

In [ ]:
sel_year = '2018'

# Normalise ship track CSV columns to match create_animation expectations
ship_data = ship_track_csv[['seg_id', 'timestamp', 'lon', 'lat']].rename(
    columns={'seg_id': 'mmsi', 'timestamp': 'dateTime'}
).copy()
ship_data['dateTime'] = pd.to_datetime(ship_data['dateTime'])

print(f"Ship positions: {len(ship_data)}, segments: {ship_data['mmsi'].nunique()}")

anim = create_animation(
    sea_ice_datasets,
    filtered_whale_data,
    sel_year,
    ship_df=ship_data,
    whale_window=7,
    ship_window=4,
    proximity_threshold_km=10,
)

output_path = 'whale_corridors_ships_2018.mp4'
writer = animation.FFMpegWriter(fps=1.5, bitrate=2000)
anim.save(output_path, writer=writer, savefig_kwargs={'pad_inches': 0.05})
plt.close('all')

Video(output_path, embed=True)

In [ ]:
sel_year = '2018'

anim_ships_only = create_animation(
    sea_ice_datasets,
    filtered_whale_data,
    sel_year,
    ship_df=ship_data,
    ship_window=4,
    show_whales=False,
)

output_path = 'ship_corridors_2018.mp4'
writer = animation.FFMpegWriter(fps=1.5, bitrate=2000)
anim_ships_only.save(output_path, writer=writer, savefig_kwargs={'pad_inches': 0.05})
plt.close('all')

Video(output_path, embed=True)

In [ ]:
sel_year = '2010'
anim = create_animation(sea_ice_datasets, filtered_whale_data, sel_year)

output_path = 'whale_corridors_2010.mp4'
writer = animation.FFMpegWriter(fps=10, bitrate=2000)
anim.save(output_path, writer=writer, savefig_kwargs={'pad_inches': 0.05})
plt.close('all')


Video(output_path, embed=True)